In [8]:
import pandas as pd
import numpy as np

data = pd.read_csv('heart.csv')
data = pd.DataFrame(data)
data

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [9]:
data.isnull().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [10]:
x = data.drop('HeartDisease',axis=1)
y = data['HeartDisease']

In [11]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

transfer = ColumnTransformer(
    transformers=[
     ('cat',OneHotEncoder(sparse_output=False,drop='first'),categorical_cols),
],
remainder='passthrough'
)



In [12]:
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

classify = {"RandomForestClassifier":RandomForestClassifier(n_estimators=100,random_state=42),
            "SVC":SVC()}


for name,model in classify.items():
    if name == "SVC":
        # Add StandardScaler for SVC and perform GridSearchCV
        pipe = Pipeline(steps=[
            ('transformer', transfer),
            ('scaler', StandardScaler()),
            ('classifier', model)
        ])
        
        # Hyperparameter tuning for SVC
        param_grid = {
            'classifier__C': [0.1, 1, 10, 100],
            'classifier__kernel': ['linear', 'rbf', 'poly'],
            'classifier__gamma': ['scale', 'auto', 0.1, 1]
        }

        grid_search = GridSearchCV(
            pipe,
            param_grid,
            cv=5,
            scoring='accuracy',
            n_jobs=-1
        )
        grid_search.fit(x_train, y_train)

        best_pipe = grid_search.best_estimator_

        print(f"Model: {name}")
        print(f"Best Params: {grid_search.best_params_}")
        print(f"Train Score: {best_pipe.score(x_train, y_train)}")
        print(f"Test Score: {best_pipe.score(x_test, y_test)}")
        print("-" * 30)
        best_model = best_pipe
    else:
        pipe = Pipeline(steps=[
        ('transformer',transfer),
        ('classify',model)]
        )
        pipe.fit(x_train,y_train)
        print(f"Model: {name}")
        print(f"Train Score: {pipe.score(x_train,y_train)}")
        print(f"Test Score: {pipe.score(x_test,y_test)}")
        print("-"*30)

        best_model = pipe

Model: RandomForestClassifier
Train Score: 1.0
Test Score: 0.8804347826086957
------------------------------
Model: SVC
Best Params: {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Train Score: 0.8801089918256131
Test Score: 0.8586956521739131
------------------------------


In [13]:
import pickle
pickle.dump(best_model,open('model.pkl','wb'))

In [14]:

load=pickle.load(open('model.pkl','rb'))
sample = pd.DataFrame([{
    'Age': 63,
    'Sex': 'M',
    'ChestPainType': 'ATA',
    'RestingBP': 145,
    'Cholesterol': 233,
    'FastingBS': 1,
    'RestingECG': 'Normal',
    'MaxHR': 150,
    'ExerciseAngina': 'N',
    'Oldpeak': 2.3,
    'ST_Slope': 'Up'
}])
load.predict(sample)


array([0])